# MDS — QUBO + Simulated Annealing

Mismo QUBO que QAOA/VQE ($A\sum x + B\sum(1-\sum_{N[v]}x)^2$). Metaheurística
**clásica** sobre el espacio de bits.


In [1]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


In [2]:
nodes = ["Mario", "Sarah", "Raul", "Ana", "Enrique", "Saul"]
edges = [("Mario", "Sarah"), ("Mario", "Raul"), ("Mario", "Ana"),
         ("Enrique", "Sarah"), ("Enrique", "Raul"), ("Saul", "Ana")]
idx = {v: i for i, v in enumerate(nodes)}
n = len(nodes)
adj = {i: set() for i in range(n)}
for a, b in edges:
    adj[idx[a]].add(idx[b])
    adj[idx[b]].add(idx[a])
closed = {v: sorted(adj[v] | {v}) for v in range(n)}

def bits_cost_payments(bits):
    return int(sum(bits))

def is_dominating(bits):
    return all(any(bits[u] for u in closed[v]) for v in range(n))

def decode(bits):
    return [nodes[i] for i in range(n) if bits[i]]

# Óptimo exacto (referencia compartida)
best_cost, best_bits = n + 1, None
for k in range(2 ** n):
    bits = [(k >> i) & 1 for i in range(n)]
    if is_dominating(bits):
        c = bits_cost_payments(bits)
        if c < best_cost:
            best_cost, best_bits = c, bits
OPTIMAL_COST = float(best_cost)
OPTIMAL_SOL = decode(best_bits)
print("Óptimo MDS:", OPTIMAL_SOL, "coste =", OPTIMAL_COST)

# QUBO MDS (Lucas-style): A*sum x + B*sum (1 - sum_{N[v]} x)^2
A_PEN, B_PEN = 1.0, 2.0
const = 0.0
lin = np.zeros(n)
quad = {}
for i in range(n):
    lin[i] += A_PEN
for v in range(n):
    L = closed[v]
    const += B_PEN
    for u in L:
        lin[u] -= B_PEN
    for a in range(len(L)):
        for b in range(a + 1, len(L)):
            key = tuple(sorted((L[a], L[b])))
            quad[key] = quad.get(key, 0.0) + 2 * B_PEN

def qubo_cost(bits):
    val = const
    for i in range(n):
        val += lin[i] * bits[i]
    for (i, j), c in quad.items():
        val += c * bits[i] * bits[j]
    return float(val)

def resolve_metrics_dir() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        root = base / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization"
        if root.is_dir():
            d = root / "metrics"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = Path.cwd().resolve() / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization" / "metrics"
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_mds_metrics(payload: dict, filename: str):
    path = resolve_metrics_dir() / filename
    path.write_text(json.dumps(payload, indent=2))
    print("Saved:", path)
    return path


Óptimo MDS: ['Ana', 'Enrique'] coste = 2.0


## Simulated Annealing


In [3]:
def sa_mds(seed=0, max_iter=4000, T0=5.0, cooling=0.995):
    random.seed(seed)
    bits = [random.randint(0, 1) for _ in range(n)]
    e = qubo_cost(bits)
    best_bits, best_e = bits[:], e
    T = T0
    evals = 1
    for _ in range(max_iter):
        nb = bits[:]
        j = random.randrange(n)
        nb[j] = 1 - nb[j]
        en = qubo_cost(nb)
        evals += 1
        d = en - e
        if d < 0 or random.random() < math.exp(-d / T):
            bits, e = nb, en
            if e < best_e:
                best_bits, best_e = bits[:], e
        T *= cooling
    return best_bits, best_e, evals

N_RUNS = 20
rows = []
for seed in range(N_RUNS):
    t0 = time.perf_counter()
    bits, energy, evals = sa_mds(seed=seed)
    elapsed = time.perf_counter() - t0
    # interpretación: coste de negocio = nº pagos si factible; si no, coste QUBO
    feasible = is_dominating(bits)
    pay = bits_cost_payments(bits)
    # success = dominating and optimal cardinality
    success = feasible and abs(pay - OPTIMAL_COST) < 1e-9
    rows.append({
        "seed": seed,
        "solution": decode(bits),
        "cost": float(pay) if feasible else float(energy),
        "payments": int(pay),
        "qubo_energy": float(energy),
        "feasible": feasible,
        "success": bool(success),
        "gap": float(pay - OPTIMAL_COST) if feasible else None,
        "time_s": float(elapsed),
        "energy_evals": evals,
    })

success_rate = float(np.mean([r["success"] for r in rows]))
# report payment costs for feasible runs
pay_costs = np.array([r["payments"] for r in rows if r["feasible"]], dtype=float)
times = np.array([r["time_s"] for r in rows], dtype=float)
print(f"success_rate={success_rate:.3f}")

payload = {
    "problem": "mds",
    "solver": "simulated_annealing",
    "family": "classical",
    "protocol": {"n_runs": N_RUNS, "max_iter": 4000, "T0": 5.0, "cooling": 0.995, "A": A_PEN, "B": B_PEN},
    "instance": {"nodes": nodes, "edges": [list(e) for e in edges]},
    "quality": {
        "optimal_cost": OPTIMAL_COST,
        "optimal_solution": OPTIMAL_SOL,
        "cost_mean": float(np.mean(pay_costs)) if len(pay_costs) else None,
        "cost_best": float(np.min(pay_costs)) if len(pay_costs) else None,
        "gap_mean": float(np.mean(pay_costs - OPTIMAL_COST)) if len(pay_costs) else None,
    },
    "success_rate": success_rate,
    "time_s": {"mean": float(np.mean(times)), "std": float(np.std(times)), "total": float(np.sum(times))},
    "effort": {"energy_evals_mean": float(np.mean([r["energy_evals"] for r in rows]))},
    "runs": rows,
}
save_mds_metrics(payload, "sa_mds.json")


success_rate=1.000
Saved: /Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/sa_mds.json


PosixPath('/Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/sa_mds.json')

## Storytelling plots

Figuras para documento/slides → `figures/`.


In [ ]:
# --- storytelling plots (auto) ---
from pathlib import Path
import sys
import numpy as np

for _base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _lib = _base / "Notebooks" / "lib"
    if (_lib / "storytelling.py").is_file():
        _s = str(_lib)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break
else:
    raise FileNotFoundError("Notebooks/lib/storytelling.py not found")

from storytelling import (
    resolve_figures_dir,
    plot_mds_instance, plot_mds_solution,
)

figures_dir = resolve_figures_dir("Optimization", "MinimumDominatingSetOptimization")
plot_mds_instance(nodes, edges, figures_dir, stem="instance")
_dom = None
if "payload" in globals() and payload.get("runs"):
    _runs = [r for r in payload["runs"] if r.get("solution") is not None]
    if _runs:
        _dom = min(_runs, key=lambda r: float(r.get("cost", 1e18))).get("solution")
if _dom is None and "sol" in globals():
    _dom = sol
if _dom is None and "OPTIMAL_SOL" in globals():
    _dom = OPTIMAL_SOL
if _dom is None and "best_b" in globals() and best_b is not None:
    _dom = decode(best_b)
if _dom is None:
    raise RuntimeError("No MDS dominating set found for storytelling plot")
plot_mds_solution(nodes, edges, _dom, figures_dir, stem="sa", title="Simulated Annealing")
